# Mini-Projet : Planification Robuste sur Grille
## A* + Chaînes de Markov

**Objectif** : Combiner la recherche heuristique (A*, UCS, Greedy) et les chaînes de Markov  
pour planifier un chemin optimal et évaluer sa robustesse face à l'incertitude stochastique.

---

## 0. Configuration et imports

In [2]:
import matplotlib
matplotlib.use('Agg')
import os, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator
import warnings
warnings.filterwarnings("ignore")

# Modules du projet
from grid       import GRIDS, visualize_grid
from astar      import ucs, greedy, astar, h_manhattan, h_zero, path_to_policy, print_result
from markov     import (build_markov, evolve, prob_goal_over_time,
                        communication_classes, classify_states, absorption_analysis)
from simulation import simulate_trajectories, print_simulation_report

OUTPUT_DIR = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

ALGO_COLORS = {"UCS": "#E53935", "Greedy": "#FB8C00", "A*": "#1E88E5"}
print("✓ Imports OK — dossier outputs :", OUTPUT_DIR)

✓ Imports OK — dossier outputs : c:\Users\Hp\OneDrive\Bureau\mon_proje\outputs


---
## Expérience 1 — Comparaison UCS / Greedy / A* sur 3 grilles

On compare les trois algorithmes sur les grilles **facile** (5×5), **moyenne** (7×7)  
et **difficile** (10×10) selon : coût du chemin, nœuds développés, temps d'exécution.

Les trois chemins sont superposés sur chaque grille :
- 🔴 **UCS** : uniforme, explore tout
- 🟠 **Greedy** : rapide mais pas toujours optimal
- 🔵 **A*** : optimal + efficace

In [3]:
def draw_multi_paths(grid, start, goal, paths_dict, title, ax):
    rows, cols = len(grid), len(grid[0])
    ax.set_facecolor("#F8F9FA")
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color="#1a1a2e", zorder=1))
    ax.set_xticks(np.arange(-0.5, cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, rows, 1), minor=True)
    ax.grid(which="minor", color="#cccccc", linewidth=0.6, zorder=0)
    ax.tick_params(which="minor", size=0)
    ax.set_xticks(range(cols)); ax.set_yticks(range(rows))
    offsets = {"UCS": (-0.13, 0), "Greedy": (0, 0.13), "A*": (0.13, 0)}
    for algo, path in paths_dict.items():
        color = ALGO_COLORS[algo]; ox, oy = offsets[algo]
        for (r, c) in path:
            if (r, c) not in (start, goal):
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color=color, alpha=0.10, zorder=2))
        for i in range(len(path)-1):
            r1, c1 = path[i]; r2, c2 = path[i+1]
            ax.annotate("", xy=(c2+ox, r2+oy), xytext=(c1+ox, r1+oy),
                        arrowprops=dict(arrowstyle="->", color=color, lw=1.8), zorder=4)
    ax.add_patch(plt.Circle((start[1], start[0]), 0.35, color="#2E7D32", zorder=5))
    ax.add_patch(plt.Circle((goal[1], goal[0]),   0.35, color="#C62828", zorder=5))
    ax.text(start[1], start[0], "S", ha="center", va="center", fontsize=9, color="white", fontweight="bold", zorder=6)
    ax.text(goal[1],  goal[0],  "G", ha="center", va="center", fontsize=9, color="white", fontweight="bold", zorder=6)
    ax.set_xlim(-0.5, cols-0.5); ax.set_ylim(rows-0.5, -0.5)
    ax.set_title(title, fontsize=9, fontweight="bold", pad=5)
print("✓ Fonction draw_multi_paths définie")

✓ Fonction draw_multi_paths définie


In [5]:
summary = []
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.patch.set_facecolor("#FAFAFA")
fig.suptitle("Expérience 1 — UCS / Greedy / A* sur les 3 grilles",
             fontsize=14, fontweight="bold", y=1.01)

for ax, name in zip(axes, ["facile", "moyenne", "difficile"]):
    cfg = GRIDS[name]; grid = cfg["grid"]; start = cfg["start"]; goal = cfg["goal"]
    r_ucs = ucs(grid, start, goal)
    r_grd = greedy(grid, start, goal)
    r_ast = astar(grid, start, goal)
    summary.append({"name": name, "ucs": r_ucs, "greedy": r_grd, "astar": r_ast})
    paths = {"UCS": r_ucs["path"], "Greedy": r_grd["path"], "A*": r_ast["path"]}
    title = f"Grille {name}\nUCS={r_ucs['cost']:.0f}  Greedy={r_grd['cost']:.0f}  A*={r_ast['cost']:.0f}"
    draw_multi_paths(grid, start, goal, paths, title, ax)

patches = [mpatches.Patch(color=ALGO_COLORS[a], label=a) for a in ["UCS","Greedy","A*"]]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=11,
           frameon=True, fancybox=True, shadow=True, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "exp1_grilles.png"), dpi=150, bbox_inches="tight", facecolor="#FAFAFA")
plt.show()
print("✓ Image exp1_grilles.png sauvegardée")

✓ Image exp1_grilles.png sauvegardée


In [7]:
# Métriques comparatives
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric, label in zip(axes,
        ["cost","nodes_dev","time_s"],
        ["Coût du chemin","Nœuds développés","Temps (s)"]):
    x = np.arange(len(summary)); w = 0.25
    ax.bar(x-w, [s["ucs"][metric]   for s in summary], w, label="UCS",    color="#E53935")
    ax.bar(x,   [s["greedy"][metric] for s in summary], w, label="Greedy", color="#FB8C00")
    ax.bar(x+w, [s["astar"][metric]  for s in summary], w, label="A*",     color="#1E88E5")
    ax.set_xticks(x); ax.set_xticklabels([s["name"] for s in summary])
    ax.set_title(label); ax.legend(); ax.grid(axis="y", alpha=0.3)
    ax.yaxis.set_major_locator(MaxNLocator(integer=(metric != "time_s")))
fig.suptitle("Expérience 1 — Comparaison des métriques", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "exp1_comparaison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✓ Image exp1_comparaison.png sauvegardée")

✓ Image exp1_comparaison.png sauvegardée


---
## Expérience 2 — Impact de ε sur P(GOAL)

Pour ε ∈ {0.0, 0.1, 0.2, 0.3}, on construit la matrice de transition P  
et on calcule π⁽ⁿ⁾ = π⁽⁰⁾ · Pⁿ pour comparer avec la simulation Monte-Carlo.

In [8]:
cfg = GRIDS["moyenne"]; grid = cfg["grid"]; start = cfg["start"]; goal = cfg["goal"]
res_astar = astar(grid, start, goal)
policy = path_to_policy(res_astar["path"], grid)
epsilons = [0.0, 0.1, 0.2, 0.3]; N_steps = 60

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Expérience 2 — P(GOAL) selon ε", fontsize=13, fontweight="bold")
axes = axes.flatten()

print(f"{'ε':>6}  {'P_sim':>10}  {'P_mat':>10}  {'E[T]':>8}")
for ax, eps in zip(axes, epsilons):
    states, idx, P = build_markov(grid, policy, goal, epsilon=eps)
    pi0 = np.zeros(len(states)); pi0[idx[start]] = 1.0
    dists = evolve(pi0, P, N_steps)
    pg_mat = prob_goal_over_time(dists, idx)
    sim_res = simulate_trajectories(states, idx, P, start, N_traj=3000, max_steps=N_steps)
    print(f"{eps:>6.1f}  {sim_res['prob_goal']:>10.4f}  {pg_mat[-1]:>10.4f}  {sim_res['mean_time_goal']:>8.2f}")
    steps = np.arange(N_steps + 1)
    ax.plot(steps, pg_mat, lw=2, color="#2196F3", label="Calcul P^n")
    ax.axhline(sim_res["prob_goal"], color="#FF5722", ls="--", lw=1.5,
               label=f"Sim. MC ({sim_res['prob_goal']:.3f})")
    ax.fill_between(steps, pg_mat, alpha=0.15, color="#2196F3")
    ax.set_xlabel("Étapes n"); ax.set_ylabel("P(GOAL)")
    ax.set_title(f"ε = {eps}"); ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "exp2_epsilon.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✓ Image exp2_epsilon.png sauvegardée")

     ε       P_sim       P_mat      E[T]
   0.0      1.0000      1.0000     12.00
   0.1      0.9113      0.9031     16.23
   0.2      0.8487      0.8505     20.80
   0.3      0.7833      0.7788     26.03
✓ Image exp2_epsilon.png sauvegardée


---
## Expérience 3 — Heuristiques : h=0 vs Manhattan vs WA*

| Heuristique | Admissible | Cohérente | Remarque |
|-------------|-----------|-----------|---------|
| h = 0 | ✓ | ✓ | Dégénère en UCS |
| Manhattan | ✓ | ✓ | Optimal et efficace |
| WA* (w=2) | ✗ | ✗ | Plus rapide, sous-optimal borné |

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Expérience 3 — Nœuds développés selon l'heuristique", fontsize=13, fontweight="bold")

for ax, name in zip(axes, ["facile", "moyenne", "difficile"]):
    cfg = GRIDS[name]; grid = cfg["grid"]; start = cfg["start"]; goal = cfg["goal"]
    r_h0 = astar(grid, start, goal, heuristic=h_zero,      w=1.0)
    r_h1 = astar(grid, start, goal, heuristic=h_manhattan, w=1.0)
    r_w2 = astar(grid, start, goal, heuristic=h_manhattan, w=2.0)
    heuristics = ["A* (h=0)", "A* (Manhattan)", "WA* (w=2)"]
    nodes = [r_h0["nodes_dev"], r_h1["nodes_dev"], r_w2["nodes_dev"]]
    costs = [r_h0["cost"],      r_h1["cost"],      r_w2["cost"]]
    bars = ax.bar(heuristics, nodes, color=["#9C27B0","#4CAF50","#FF9800"], alpha=0.85)
    for bar, cost in zip(bars, costs):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"coût={cost:.0f}", ha="center", va="bottom", fontsize=9)
    ax.set_title(f"Grille {name}"); ax.set_ylabel("Nœuds développés"); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "exp3_heuristiques.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✓ Image exp3_heuristiques.png sauvegardée")

✓ Image exp3_heuristiques.png sauvegardée


---
## Expérience 4 — Analyse Markov : classes, absorption, simulation

- **Classes de communication** : identification par algorithme de Kosaraju
- **Absorption** : matrice fondamentale N = (I − Q)⁻¹, probabilités B = N·R, temps t = N·1
- **Monte-Carlo** : 5000 trajectoires simulées

In [10]:
cfg = GRIDS["facile"]; grid = cfg["grid"]; start = cfg["start"]; goal = cfg["goal"]
res = astar(grid, start, goal)
pol = path_to_policy(res["path"], grid)
eps = 0.15
states, idx, P = build_markov(grid, pol, goal, epsilon=eps)
n = len(states)

# Classes
classes = communication_classes(P, states)
class_info = classify_states(P, states, classes)
print(f"Grille FACILE | ε={eps} | {n} états")
print(f"Classes de communication : {len(classes)}")
for ci in class_info:
    ptype = "PERSISTANT (absorbant)" if ci["absorbing"] else ("PERSISTANT" if ci["persistent"] else "TRANSITOIRE")
    print(f"  Classe {ci['class_id']} ({ci['size']} états) → {ptype}")
    if ci["size"] <= 8: print(f"    {ci['states']}")

# Absorption
ab = absorption_analysis(P, states, idx, absorbing_states=["GOAL"])
if ab and ab["N"] is not None:
    print("\nAbsorption vers GOAL :")
    for s in res["path"][:6]:
        if s in idx and s in ab["trans_states"]:
            ti = ab["trans_states"].index(s)
            print(f"  {s} → P(GOAL)={ab['B'][ti,0]:.4f}  E[T]={ab['t'][ti]:.2f}")

Grille FACILE | ε=0.15 | 22 états
Classes de communication : 3
  Classe 0 (1 états) → TRANSITOIRE
    [(4, 4)]
  Classe 1 (20 états) → TRANSITOIRE
  Classe 2 (1 états) → PERSISTANT (absorbant)
    ['GOAL']

Absorption vers GOAL :
  (0, 0) → P(GOAL)=1.0000  E[T]=23.82
  (1, 0) → P(GOAL)=1.0000  E[T]=17.44
  (2, 0) → P(GOAL)=1.0000  E[T]=15.89
  (3, 0) → P(GOAL)=1.0000  E[T]=10.81
  (4, 0) → P(GOAL)=1.0000  E[T]=9.33
  (4, 1) → P(GOAL)=1.0000  E[T]=8.07


In [11]:
sim = simulate_trajectories(states, idx, P, start, N_traj=5000, max_steps=200)
print_simulation_report(sim)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Expérience 4 — Analyse Markov (grille facile, ε={eps})", fontweight="bold")
axes[0].hist(sim["times_goal"], bins=25, color="#42A5F5", edgecolor="white", density=True)
axes[0].axvline(sim["mean_time_goal"], color="red", lw=2, label=f"Moy={sim['mean_time_goal']:.1f}")
axes[0].set_xlabel("Étapes"); axes[0].set_ylabel("Densité")
axes[0].set_title("Temps d'atteinte GOAL"); axes[0].legend(); axes[0].grid(alpha=0.3)

pi0 = np.zeros(n); pi0[idx[start]] = 1.0
dists = evolve(pi0, P, 100); pg_mat = prob_goal_over_time(dists, idx)
axes[1].plot(np.arange(101), pg_mat, lw=2.5, color="#1565C0", label="P^n")
axes[1].set_xlabel("Étapes n"); axes[1].set_ylabel("P(GOAL)")
axes[1].set_title("Convergence vers GOAL"); axes[1].grid(alpha=0.3)
axes[1].legend(); axes[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "exp4_markov.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✓ Image exp4_markov.png sauvegardée")

  Trajectoires simulées : 5000
  P(atteindre GOAL)     : 0.9748
  Temps moyen (si GOAL) : 16.11 ± 23.07 étapes
  P(bloqué / timeout)   : 0.0252
✓ Image exp4_markov.png sauvegardée


In [12]:
cfg = GRIDS["facile"]; grid = cfg["grid"]; start = cfg["start"]; goal = cfg["goal"]
res = astar(grid, start, goal)
pol = path_to_policy(res["path"], grid)
epsilons = [0.0, 0.1, 0.2, 0.3]

fig, axes = plt.subplots(2, 2, figsize=(22, 18))
fig.patch.set_facecolor("#0b1120")
fig.suptitle("Matrices de Transition  P(i, j)  pour chaque ε\n"
             "Grille facile 5×5  —  politique A*  —  Σ lignes = 1",
             fontsize=15, fontweight="bold", color="white", y=0.99)

for ax, eps in zip(axes.flatten(), epsilons):
    states_e, idx_e, P_e = build_markov(grid, pol, goal, epsilon=eps)
    n_e = len(states_e)
    labels = [s if s == "GOAL" else str(s) for s in states_e]
    masked = np.ma.masked_where(P_e == 0, P_e)
    cmap = plt.colormaps["YlOrRd"].with_extremes(bad="#0b1a30")
    im = ax.imshow(masked, cmap=cmap, vmin=0, vmax=1.0, aspect="equal")
    for i in range(n_e):
        for j in range(n_e):
            v = P_e[i, j]
            if v == 0: continue
            txt = "P(1)" if v == 1.0 else f"P({v:.3f})"
            fc = "white" if v > 0.45 else ("#333" if v < 0.15 else "#111")
            ax.text(j, i, txt, ha="center", va="center", fontsize=5.5, color=fc,
                    fontweight="bold" if v >= 0.7 else "normal")
    ax.set_xticks(range(n_e)); ax.set_yticks(range(n_e))
    ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=7, color="#aabbdd")
    ax.set_yticklabels(labels, fontsize=7, color="#aabbdd")
    ax.tick_params(colors="#aabbdd", length=0); ax.set_facecolor("#0b1a30")
    for sp in ax.spines.values(): sp.set_edgecolor("#223366"); sp.set_linewidth(1.5)
    row_sums = P_e.sum(axis=1)
    all_ok = np.allclose(row_sums, 1.0, atol=1e-9)
    for i in range(n_e):
        color = "#44dd88" if abs(row_sums[i]-1) < 1e-9 else "#ff4444"
        ax.text(n_e, i, f"={row_sums[i]:.2f}", va="center", ha="left",
                fontsize=5.5, color=color, fontweight="bold")
    ax.set_xlim(-0.5, n_e + 1.0)
    ax.set_title(f"P(i, j)  avec  ε = {eps}     Σ = 1 {'✓' if all_ok else '✗'}",
                 color="#7eb8ff", fontsize=11, fontweight="bold", pad=10)
    cb = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.ax.tick_params(labelsize=7, colors="#aabbdd")
    print(f"  ε={eps} : {n_e}×{n_e}  Σ=1 {'✓' if all_ok else '✗'}")

plt.tight_layout(rect=[0, 0, 1, 0.97], h_pad=3, w_pad=2)
plt.savefig(os.path.join(OUTPUT_DIR, "exp5_matrices_P.png"), dpi=150, bbox_inches="tight", facecolor="#0b1120")
plt.show()
print("✓ Image exp5_matrices_P.png sauvegardée")

  ε=0.0 : 22×22  Σ=1 ✓
  ε=0.1 : 22×22  Σ=1 ✓
  ε=0.2 : 22×22  Σ=1 ✓
  ε=0.3 : 22×22  Σ=1 ✓
✓ Image exp5_matrices_P.png sauvegardée


---
## Affichage Terminal — Matrice P(i,j) sous forme tableau

Format lisible avec `·` pour les zéros et vérification Σ = 1 par ligne.

In [13]:
def afficher_matrice_P(epsilon=0.1, grille="facile"):
    cfg = GRIDS[grille]; grid = cfg["grid"]; start = cfg["start"]; goal = cfg["goal"]
    res = astar(grid, start, goal)
    policy = path_to_policy(res["path"], grid)
    states, idx, P = build_markov(grid, policy, goal, epsilon=epsilon)
    n = len(states); labels = [str(s) for s in states]; W = 8
    print("\n" + "="*60)
    print(f"  MATRICE P(i,j)  |  ε = {epsilon}  |  grille = {grille}")
    print(f"  {n} états  —  Σ lignes = 1")
    print("="*60)
    header = f"{'':>12}" + "".join(f"{l:>{W}}" for l in labels) + "   | SOMME"
    print(header); print("-" * len(header))
    for i in range(n):
        row_str = f"{labels[i]:>12}"
        for v in P[i]:
            if v == 0.0:   row_str += f"{'·':>{W}}"
            elif v == 1.0: row_str += f"{'1':>{W}}"
            else:           row_str += f"{v:{W}.4f}"
        total = P[i].sum()
        ok = "✓" if abs(total - 1.0) < 1e-9 else "✗"
        row_str += f"   | {total:.4f} {ok}"
        print(row_str)
    print("-" * len(header))
    print(f"  Toutes les sommes = 1 : {'✓' if np.allclose(P.sum(axis=1), 1) else '✗'}")

# Afficher pour ε = 0.1
afficher_matrice_P(epsilon=0.1, grille="facile")


  MATRICE P(i,j)  |  ε = 0.1  |  grille = facile
  22 états  —  Σ lignes = 1
              (0, 0)  (0, 1)  (0, 2)  (0, 3)  (0, 4)  (1, 0)  (1, 3)  (1, 4)  (2, 0)  (2, 1)  (2, 2)  (2, 3)  (2, 4)  (3, 0)  (3, 2)  (3, 4)  (4, 0)  (4, 1)  (4, 2)  (4, 3)  (4, 4)    GOAL   | SOMME
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
      (0, 0)  0.0667  0.0333       ·       ·       ·  0.9000       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·   | 1.0000 ✓
      (0, 1)  0.0250  0.9500  0.0250       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·   | 1.0000 ✓
      (0, 2)       ·  0.0250  0.9500  0.0250       ·       ·       ·       ·       ·       ·       ·       ·       ·    

In [14]:
# Afficher pour tous les epsilon
for eps in [0.0, 0.1, 0.2, 0.3]:
    afficher_matrice_P(epsilon=eps, grille="facile")


  MATRICE P(i,j)  |  ε = 0.0  |  grille = facile
  22 états  —  Σ lignes = 1
              (0, 0)  (0, 1)  (0, 2)  (0, 3)  (0, 4)  (1, 0)  (1, 3)  (1, 4)  (2, 0)  (2, 1)  (2, 2)  (2, 3)  (2, 4)  (3, 0)  (3, 2)  (3, 4)  (4, 0)  (4, 1)  (4, 2)  (4, 3)  (4, 4)    GOAL   | SOMME
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
      (0, 0)       ·       ·       ·       ·       ·       1       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·   | 1.0000 ✓
      (0, 1)       ·       1       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·   | 1.0000 ✓
      (0, 2)       ·       ·       1       ·       ·       ·       ·       ·       ·       ·       ·       ·       ·    

---
## Récapitulatif des fichiers générés

| Fichier | Contenu |
|---------|---------|
| `exp1_grilles.png` | Chemins UCS / Greedy / A* superposés sur 3 grilles |
| `exp1_comparaison.png` | Métriques comparatives (coût, nœuds, temps) |
| `exp2_epsilon.png` | P(GOAL,n) pour ε ∈ {0, 0.1, 0.2, 0.3} |
| `exp3_heuristiques.png` | Nœuds développés selon h=0, Manhattan, WA* |
| `exp4_markov.png` | Distribution temps d'atteinte + convergence P^n |


---
*Mini-projet — A* + Chaînes de Markov — 2025-2026*

In [16]:
print("\n=== Images générées dans outputs/ ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".png"):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) // 1024
        print(f"  • {f}  ({size} KB)")


=== Images générées dans outputs/ ===
  • exp1_comparaison.png  (63 KB)
  • exp1_grilles.png  (120 KB)
  • exp2_epsilon.png  (110 KB)
  • exp3_heuristiques.png  (68 KB)
  • exp4_markov.png  (59 KB)
